In [1]:
import numpy as np
import random as rd
import copy
import psutil
import gc
import pickle
from qiskit.quantum_info import SparsePauliOp
import time as time_lib
n_qubit = 1
dim     = 2**n_qubit

In [2]:
def memory_usage(message: str = 'debug'):
    # this memory_usage function is imported from https://jybaek.tistory.com/895
    # current process RAM usage
    p = psutil.Process()
    rss = p.memory_info().rss / 2 ** 30 # Bytes to GiB
    print(f"[{message}] memory usage: {rss: 10.5f} GiB")

In [3]:
x_op = SparsePauliOp.from_list([('X',1.0)])
y_op = SparsePauliOp.from_list([('Y',1.0)])
z_op = SparsePauliOp.from_list([('Z',1.0)])
x_mat = x_op.to_matrix()
y_mat = y_op.to_matrix()
z_mat = z_op.to_matrix()

In [4]:
from qiskit import transpile, Aer
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_aer.primitives import Sampler as AerSampler

estimator = AerEstimator()

nshot    = 4000

In [5]:
q_layout = [0,1]
z_hadamard_test = SparsePauliOp.from_sparse_list([('Z',[q_layout[0]],1)],num_qubits=n_qubit+1)
print(z_hadamard_test)

SparsePauliOp(['IZ'],
              coeffs=[1.+0.j])


In [6]:
from qiskit import QuantumRegister
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector

qr = QuantumRegister(n_qubit+1, 'q')
params = ParameterVector('θ',5)

In [7]:
t_x          = 0.5
nmc          = int(400)


In [8]:
nld_list  =  [3, 5, 7, 9, 11, 16, 21, 26, 31, 36, 41, 46, 51]
n_nld = len(nld_list)
print(n_nld)

13


In [9]:
# run qzmc;
norms_nld = [[] for _ in range(n_nld)]
eigen_energies_nld = [[] for _ in range(n_nld)]
for i_nld in range(n_nld):
    nld = nld_list[i_nld]
    norms_nld[i_nld]            = np.ones((nld),dtype=float)
    eigen_energies_nld[i_nld]   = np.zeros((nld),dtype=float)

In [10]:
result_values_nld   = [[[] for _ in range(nld_list[i_nld])] for i_nld in range(n_nld)]

In [14]:
memory_usage('before run')

[before run] memory usage:    0.18947 GiB


In [15]:
# observable amplitudes
n_obs = 3
# 3 different observables are computed
# 0; norm, 1; dE1, 2; Z

In [16]:
beta = 5.0

In [17]:
i_state = 0 # only consider the ground state
for i_nld in range(n_nld):
    nld = nld_list[i_nld]

    t_zs         = []
    hamiltonians = []
    alpha        = 0
    lds     = np.linspace(0,1,num=nld)
    for ld in lds:
        t_z = -1 + 2.0 * ld
        h = SparsePauliOp.from_list([('X',t_x),('Z',t_z)])
        t_zs.append(t_z)
        hamiltonians.append(h)
        alpha += 1
    hamiltonian_diffs = []
    for ild in range(nld-1):
        hamiltonian_diffs.append((hamiltonians[ild+1]-hamiltonians[ild]).simplify())
    hamiltonian_diffs_list = []
    for ild in range(nld-1):
        hamiltonian_diffs_list.append(hamiltonian_diffs[ild].to_list())
    
    n_hamiltonians = len(hamiltonians)
    # exact eigenvalues
    eigen_energies_exact  = np.zeros((nld,dim),dtype=float)
    eigen_vectors_exact   = np.zeros((nld,dim,dim),dtype=complex)
    for ild in range(nld):
        eigen_e, eigen_v = np.linalg.eigh(hamiltonians[ild].to_matrix())
        indx = np.argsort(eigen_e.real)
        for i in range(dim):
            eigen_energies_exact[ild,i]   = eigen_e[indx[i]].real
            eigen_vectors_exact[ild,:,i] = eigen_v[:,indx[i]]
    
    def ComputeUnitaryParams(U):
        theta = 2.0 * np.arccos(np.abs(U[0,0]))
        if (theta<1E-6):
            theta = 2.0*np.arcsin(np.abs(U[0,1]))
        gamma = np.angle(U[0,0])
        if (theta<1E-10):
            phi   = np.angle(U[1,1]/U[0,0])
            lam   = 0
        else:
            phi   = np.angle(U[1,0]/np.sin(theta/2)) - gamma
            lam   = np.angle(-U[0,1]/np.sin(theta/2)) - gamma
        return [theta, phi, lam, gamma]
    
    def ExactTimeEvolution (alpha, time):
        Vl = copy.deepcopy(eigen_vectors_exact[alpha,:,:])
        evol = np.zeros((dim,dim),dtype=complex)
        exp_d = np.diag(np.exp(-1j*eigen_energies_exact[alpha,:]*time))
        evol = Vl@exp_d@Vl.conj().T
        return evol
    
    def TrotterTimeEvolution (alpha, Nt, time):
        dtime  = time/Nt
        dtx = dtime * t_x
        dtz = dtime * t_zs[alpha]
        evol = np.identity(dim)
        for it in range(Nt):
            evol_X = np.cos(dtx)*np.identity(dim) - 1j*np.sin(dtx) * x_mat
            evol_Z = np.cos(dtz)*np.identity(dim) - 1j*np.sin(dtz) * z_mat
            evol = evol_Z@evol
            evol = evol_X@evol
        return evol
    
    theta_init = np.zeros((dim),dtype=float)
    for i in range(dim):
        theta_init[i] = 2.0 * np.angle(eigen_vectors_exact[0,0,i]+ 1j*eigen_vectors_exact[0,1,i])
    def Circuit_Initialize(circuit_, i_, qr_):
        # initialize qr_[1:] to i_ th eigenvector
        circuit_.ry(theta_init[i_],qr_[1])

    eigen_energies_nld[i_nld][0] = eigen_energies_exact[0,i_state]


    O_timelists         = [[[] for _ in range(n_obs)] for _ in range(nld)]
    for alpha in range(1,nld):
        for i_obs in range(n_obs):
            for imc in range(nmc):
                times = []
                for alpha_ in range(1,alpha):
                    time = rd.gauss(0.0, beta)
                    times.append(time)

                time = rd.gauss(0.0, beta)
                times.append(time)

                time = rd.gauss(0.0, beta)
                times.append(time)

                for alpha_ in reversed(range(1,alpha)):
                    time = rd.gauss(0.0, beta)
                    times.append(time)
                O_timelists[alpha][i_obs].append(times)

    # real part measurement
    circuit_real = QuantumCircuit(qr,name='-')
    Circuit_Initialize(circuit_real,i_state,qr)
    circuit_real.h(qr[0])
    circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
    circuit_real.p(params[4],qr[0])
    circuit_real.h(qr[0])

    eps = eigen_vectors_exact[0,:,i_state].conj().T@hamiltonians[1].to_matrix()@eigen_vectors_exact[0,:,i_state]
    eps = eps.real
    for alpha in range(1,n_hamiltonians):
        params_list = []
        # 0; norm measurement
        i_obs = 0
        for imc in range(nmc):
            U_evol = np.identity(dim,dtype=complex)
            times  = O_timelists[alpha][i_obs][imc]
            phase  = 0.0
            i_time = 0
            # construction of P
            for alpha_ in range(1,alpha):
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                phase += eigen_energies_nld[i_nld][alpha_] * time
                i_time += 1

            time = times[i_time]
            U_evol = ExactTimeEvolution(alpha,time)@U_evol
            phase += eps * time
            i_time += 1

            # construction of P^\dagger

            time = times[i_time]
            U_evol = ExactTimeEvolution(alpha,time)@U_evol
            phase += eps * time
            i_time += 1

            for alpha_ in reversed(range(1,alpha)):
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                phase += eigen_energies_nld[i_nld][alpha_] * time
                i_time += 1

            theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
            params_list.append([theta,phi,lam,gamma,phase])
        # 1; dE1 measurement
        i_obs = 1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            # pass for constant contribution
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            for imc in range(nmc):
                U_evol = np.identity(dim,dtype=complex)
                times  = O_timelists[alpha][i_obs][imc]
                phase  = 0.0
                i_time = 0
                # construction of P
                for alpha_ in range(1,alpha):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_nld[i_nld][alpha_] * time
                    i_time += 1

                # insertion of (H[alpha]-H[alpha-1])
                Mat = hamiltonian_diffs[alpha-1].paulis[ihd].to_matrix()

                U_evol = Mat@U_evol

                # construction of P; continued
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                # construction of P^\dagger

                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                for alpha_ in reversed(range(1,alpha)):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_nld[i_nld][alpha_] * time
                    i_time += 1
                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
                params_list.append([theta,phi,lam,gamma,phase])

        # 2; dE2 measurement
        i_obs = 2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            # pass for constant contribution
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            for imc in range(nmc):
                U_evol = np.identity(dim,dtype=complex)
                times  = O_timelists[alpha][i_obs][imc]
                phase  = 0.0
                i_time = 0
                # construction of P
                for alpha_ in range(1,alpha):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_nld[i_nld][alpha_] * time
                    i_time += 1

                # construction of P; continued
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                # insertion of (H[alpha+1]-H[alpha])
                Mat = hamiltonian_diffs[alpha].paulis[ihd].to_matrix()

                U_evol = Mat@U_evol

                # construction of P^\dagger

                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                for alpha_ in reversed(range(1,alpha)):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_nld[i_nld][alpha_] * time
                    i_time += 1
                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
                params_list.append([theta,phi,lam,gamma,phase])

        circuits = [circuit_real.assign_parameters({params: params_}) for params_ in params_list]

        z_hadamards = [z_hadamard_test]*len(circuits)
        estimator = AerEstimator(run_options={"shots": 4000})
        job = estimator.run(circuits,z_hadamards)
        result_values = job.result().values
        result_values_nld[i_nld][alpha] = result_values
        # compute values
        norm    = 0.0
        dE1     = 0.0
        dE2     = 0.0

        i_meas = 0
        # 0; norm
        for imc in range(nmc):
            norm += result_values_nld[i_nld][alpha][i_meas]
            i_meas += 1
        # 1; dE1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
            for imc in range(nmc):
                dE1 +=  coeff * result_values_nld[i_nld][alpha][i_meas]
                i_meas += 1

        # 2; dE2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha][ihd][1]
            for imc in range(nmc):
                dE2 += coeff * result_values_nld[i_nld][alpha][i_meas]
                i_meas += 1

        dE1     /=norm
        dE2     /=norm

        norm    /=nmc

        # add constant contributions
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                dE1 += hamiltonian_diffs_list[alpha-1][ihd][1]

        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                dE2 += hamiltonian_diffs_list[alpha][ihd][1]

        eigen_energies_nld[i_nld][alpha] = eigen_energies_nld[i_nld][alpha-1] + dE1
        norms_nld[i_nld][alpha] = norm

        if (alpha<n_hamiltonians-1):
            eps = eigen_energies_nld[i_nld][alpha] + dE2
            eps = eps.real

        print(alpha, norms_nld[i_nld][alpha], eigen_energies_nld[i_nld][alpha]-eigen_energies_exact[alpha,i_state])
        if (alpha<n_hamiltonians-1):
            print(alpha, eps, eigen_energies_exact[alpha+1,i_state])
        st = '# {i_nld}/{n_nld}: {percent}%'.format(i_nld=i_nld+1,n_nld=n_nld,percent=((alpha)/(n_hamiltonians-1)*100))

        del job
        del z_hadamards
        del circuits 
        del estimator
        collected = gc.collect()
        memory_usage(st)




/tmp/ipykernel_4039/2187034093.py:288: ComplexWarning: Casting complex values to real discards the imaginary part
  eigen_energies_nld[i_nld][alpha] = eigen_energies_nld[i_nld][alpha-1] + dE1


1 0.11884000000000013 -0.4886478393052637
1 -1.0775909561009551 -1.118033988749895
[# 1/13: 50.0%] memory usage:    0.25750 GiB
2 0.009027499999999992 5.839992629644027
[# 1/13: 100.0%] memory usage:    0.26949 GiB
1 0.9390037499999994 0.0017488600242425711
1 -0.34970106143197904 -0.5
[# 2/13: 25.0%] memory usage:    0.28565 GiB
2 0.4888650000000002 -0.04195263544948036
2 -0.5064443662954194 -0.7071067811865476
[# 2/13: 50.0%] memory usage:    0.28796 GiB
3 0.27875749999999955 -0.03727314321479969
3 -1.0501407559484808 -1.118033988749895
[# 2/13: 75.0%] memory usage:    0.28756 GiB
4 0.5400162500000003 -0.05308425139658013
[# 2/13: 100.0%] memory usage:    0.28210 GiB
1 0.9871512500000008 0.00014171296417009316
1 -0.5655287368951265 -0.6009252125773316
[# 3/13: 16.666666666666664%] memory usage:    0.28605 GiB
2 0.9332687500000005 0.0010738456632062299
2 -0.4158774758636286 -0.5
[# 3/13: 33.33333333333333%] memory usage:    0.28780 GiB
3 0.7312750000000001 0.0008910578462817953
3 -0.49

In [18]:
#with open('MDH.nld.results','rb') as file_:
#    [result_values_nld] = pickle.load(file_)

with open('MDH.nld.results','wb') as file_:
    pickle.dump([result_values_nld],file_)


In [19]:
norms_raw_nld = [[] for _ in range(n_nld)]
dEnorm_raw_nld = [[] for _ in range(n_nld)]
for i_nld in range(n_nld):
    nld = nld_list[i_nld]
    norms_raw_nld[i_nld]    = np.ones((nld,nmc),dtype=float)
    dEnorm_raw_nld[i_nld]   = np.zeros((nld,nmc),dtype=float)

In [20]:
i = 0 # only consider the ground state
for i_nld in range(n_nld):
    nld = nld_list[i_nld]
    n_hamiltonians = nld
    for alpha in range(1,n_hamiltonians):
        # compute values
        norm    = 0.0
        dE1     = 0.0
        dE2     = 0.0

        i_meas = 0
        # 0; norm
        for imc in range(nmc):
            norms_raw_nld[i_nld][alpha][imc] = result_values_nld[i_nld][alpha][i_meas]
            norm += norms_raw_nld[i_nld][alpha][imc]
            i_meas += 1
        # 1; dE1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
            for imc in range(nmc):
                dEnorm_raw_nld[i_nld][alpha][imc] +=  coeff * result_values_nld[i_nld][alpha][i_meas]
                i_meas += 1
        for imc in range(nmc):
            dE1 += dEnorm_raw_nld[i_nld][alpha][imc]

        # 2; dE2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha][ihd][1]
            for imc in range(nmc):
                dE2 += coeff * result_values_nld[i_nld][alpha][i_meas]
                i_meas += 1

        dE1     /=norm
        dE2     /=norm
        norm    /=nmc

        # add constant contributions
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                dE1 += hamiltonian_diffs_list[alpha-1][ihd][1]

        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                dE2 += hamiltonian_diffs_list[alpha][ihd][1]

        eigen_energies_nld[i_nld][alpha] = eigen_energies_nld[i_nld][alpha-1] + dE1
        norms_nld[i_nld][alpha] = norm

        print(alpha, norms_nld[i_nld][alpha], eigen_energies_nld[i_nld][alpha])
        st = '# {i_nld}/{n_nld}: {percent}%'.format(i_nld=i_nld+1,n_nld=n_nld,percent=((alpha)/(n_hamiltonians-1)*100))

        #collected = gc.collect()
        memory_usage(st)




/tmp/ipykernel_4039/1388125277.py:24: ComplexWarning: Casting complex values to real discards the imaginary part
  dEnorm_raw_nld[i_nld][alpha][imc] +=  coeff * result_values_nld[i_nld][alpha][i_meas]


1 0.11884000000000013 -1.1128585427721096
[# 1/13: 50.0%] memory usage:    0.41068 GiB
2 0.009027499999999992 -0.8844342835641342
[# 1/13: 100.0%] memory usage:    0.41068 GiB
1 0.9390037499999994 -1.0850199033428878
[# 2/13: 25.0%] memory usage:    0.41068 GiB
2 0.4888650000000002 -1.0719474804858617
[# 2/13: 50.0%] memory usage:    0.41068 GiB
3 0.27875749999999955 -1.0881416636020111
[# 2/13: 75.0%] memory usage:    0.41068 GiB
4 0.5400162500000003 -1.1222807288616214
[# 2/13: 100.0%] memory usage:    0.41068 GiB
1 0.9871512500000008 -1.0838529045442071
[# 3/13: 16.666666666666664%] memory usage:    0.41068 GiB
2 0.9332687500000005 -1.0558520741296027
[# 3/13: 33.33333333333333%] memory usage:    0.41068 GiB
3 0.7312750000000001 -1.043762983158354
[# 3/13: 50.0%] memory usage:    0.41068 GiB
4 0.6074887500000004 -1.0566285712042232
[# 3/13: 66.66666666666666%] memory usage:    0.41068 GiB
5 0.7570462499999997 -1.0851319816629632
[# 3/13: 83.33333333333334%] memory usage:    0.41068 

In [22]:
# compute standard deviations
import random as rd
std_norm = [[] for _ in range(n_nld)]
std_dE   = [[] for _ in range(n_nld)]
std_E    = [[] for _ in range(n_nld)]
for i_nld in range(n_nld):
    nld = nld_list[i_nld]
    std_norm[i_nld] = np.zeros((nld),dtype=float)
    std_dE[i_nld]   = np.zeros((nld),dtype=float)
    std_E[i_nld]    = np.zeros((nld),dtype=float)
n_boot = 1000
for i_nld in range(n_nld):
    nld = nld_list[i_nld]
    n_hamiltonians = nld
    for alpha in range(1,n_hamiltonians):
        norm_boot = np.zeros((n_boot),dtype=float)
        dE_boot   = np.zeros((n_boot),dtype=float)
        for i_boot in range(n_boot):
            norm_ = 0.0
            dE_   = 0.0
            for imc in range(nmc):
                jmc = rd.randrange(nmc)
                norm_ += norms_raw_nld[i_nld][alpha][jmc]
                dE_ += dEnorm_raw_nld[i_nld][alpha][jmc]

            dE_   = dE_/norm_
            norm_ = norm_/nmc
            norm_boot[i_boot] = norm_
            dE_boot[i_boot] = dE_
        norm_boot_mean = 0.0
        dE_boot_mean   = 0.0
        for i_boot in range(n_boot):
            norm_boot_mean += norm_boot[i_boot]
            dE_boot_mean   += dE_boot[i_boot]
        norm_boot_mean /= n_boot
        dE_boot_mean /= n_boot

        var_norm = 0.0
        var_dE = 0.0
        for i_boot in range(n_boot):
            var_norm += (norm_boot[i_boot]-norm_boot_mean)**2
            var_dE += (dE_boot[i_boot]-dE_boot_mean)**2
        var_norm /= (n_boot-1)
        var_dE    /= (n_boot-1)
        std_norm[i_nld][alpha]  = np.sqrt(var_norm)
        std_dE[i_nld][alpha]    = np.sqrt(var_dE)
        std_E[i_nld][alpha]     = std_E[i_nld][alpha-1] + var_dE
for i_nld in range(n_nld):
    nld = nld_list[i_nld]
    n_hamiltonians = nld
    for alpha in range(n_hamiltonians):
        std_E[i_nld][alpha] = np.sqrt(std_E[i_nld][alpha])

In [23]:
for i_nld in range(n_nld):
    print(nld_list[i_nld],std_norm[i_nld][-1],std_E[i_nld][-1])

3 0.025196257242674245 30.293925980848627
5 0.015349019351713797 0.003981446026717893
7 0.008464538504390964 0.0018307725158565166
9 0.007621423749173015 0.0015317776398625959
11 0.005741389486372451 0.0014770221351502373
16 0.004686176653433808 0.0013999717325337154
21 0.0038316074392550216 0.0013737927131989027
26 0.003163061236466086 0.001367018799815036
31 0.0025255322389862553 0.0013558957177146884
36 0.0018840095891410832 0.0013398590155269733
41 0.001726361111837113 0.0013116182456855495
46 0.0017102273090729373 0.0013242534841964191
51 0.001417640913581419 0.0013366213670939293


In [24]:
# save to file
with open('nld.norm.E.save','w') as file_:
    s = '# nld , norm^2, std(norm^2), Enorm, std(Enorm) '
    s += '\n'
    file_.write(s)
    for i_nld in range(n_nld):
        nld = nld_list[i_nld]
        s = '{:}'.format(nld)
        s += '  {:.16e}  {:.16e}'.format(norms_nld[i_nld][-1],std_norm[i_nld][-1])
        s += '  {:.16e}  {:.16e}'.format(eigen_energies_nld[i_nld][-1],std_E[i_nld][-1])
        print(s)
        s += '\n'
        file_.write(s)

3  9.0274999999999921e-03  2.5196257242674245e-02  -8.8443428356413423e-01  3.0293925980848627e+01
5  5.4001625000000031e-01  1.5349019351713797e-02  -1.1222807288616214e+00  3.9814460267178931e-03
7  7.8780874999999995e-01  8.4645385043909636e-03  -1.1184915375196354e+00  1.8307725158565166e-03
9  8.1940500000000061e-01  7.6214237491730149e-03  -1.1190006978177374e+00  1.5317776398625959e-03
11  8.6174250000000030e-01  5.7413894863724514e-03  -1.1204537624245521e+00  1.4770221351502373e-03
16  8.9857125000000015e-01  4.6861766534338082e-03  -1.1181655725147153e+00  1.3999717325337154e-03
21  9.1491000000000111e-01  3.8316074392550216e-03  -1.1153402453229397e+00  1.3737927131989027e-03
26  9.3368249999999930e-01  3.1630612364660859e-03  -1.1176995835984040e+00  1.3670187998150360e-03
31  9.4764999999999977e-01  2.5255322389862553e-03  -1.1192412872629320e+00  1.3558957177146884e-03
36  9.5972375000000054e-01  1.8840095891410832e-03  -1.1168787616663547e+00  1.3398590155269733e-03
41  